In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# --- Bước 1: Đọc và sắp xếp dữ liệu ---
data = pd.read_csv('data.csv', parse_dates=['date'], index_col='date')
data = data.sort_index()

# --- Bước 2: Tiền xử lý dữ liệu ---
# Chuẩn hóa dữ liệu về khoảng [0,1]
scaler = MinMaxScaler(feature_range=(0, 1))
data['value_scaled'] = scaler.fit_transform(data[['value']])

# Hàm tạo đặc trưng dạng lag: dùng 3 giá trị trước để dự báo giá trị hiện tại
def create_lag_features(df, lags=3):
    df_features = pd.DataFrame()
    for lag in range(1, lags + 1):
        df_features[f'lag_{lag}'] = df['value_scaled'].shift(lag)
    df_features['target'] = df['value_scaled']
    return df_features.dropna()

lags = 3
df_features = create_lag_features(data, lags)

# Tách đặc trưng (X) và nhãn (y)
X = df_features.drop('target', axis=1).values
y = df_features['target'].values

# --- Bước 3: Chia dữ liệu thành tập huấn luyện và tập kiểm tra (80%-20%) ---
# Lưu ý: Không xáo trộn dữ liệu để giữ thứ tự thời gian
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# --- Bước 4: Xây dựng và huấn luyện mô hình SVR ---
# Sử dụng kernel RBF, có thể điều chỉnh các tham số C và epsilon
svr_model = SVR(kernel='rbf', C=100, epsilon=0.01)
svr_model.fit(X_train, y_train)

# Dự báo trên tập kiểm tra
y_pred = svr_model.predict(X_test)

# Tính toán MSE
mse = mean_squared_error(y_test, y_pred)
print(f"MSE: {mse:.4f}")

# --- Bước 5: Chuyển đổi dự báo về giá trị ban đầu ---
y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))
y_pred_inv = scaler.inverse_transform(y_pred.reshape(-1, 1))

# Trực quan hóa kết quả
plt.figure(figsize=(8, 5))
plt.plot(y_test_inv, label='Giá trị thực', marker='o')
plt.plot(y_pred_inv, label='Dự báo', marker='x', linestyle='--')
plt.title('Dự báo chuỗi thời gian với SVR')
plt.xlabel('Mẫu')
plt.ylabel('Giá trị')
plt.legend()
plt.show()

 Giải thích chi tiết các bước:
Đọc và sắp xếp dữ liệu:

Dữ liệu được đọc từ file data.csv với cột ngày được parse và sắp xếp theo thứ tự thời gian.

Tiền xử lý:

Sử dụng MinMaxScaler để chuẩn hóa cột value về khoảng [0,1].

Hàm create_lag_features tạo ra các cột lag (lag_1, lag_2, lag_3) làm đầu vào và target là giá trị hiện tại.

Các dòng dữ liệu có giá trị thiếu (do tạo lag) được loại bỏ.

Chia dữ liệu:

Dữ liệu được chia thành tập huấn luyện và kiểm tra theo tỷ lệ 80%-20% mà không xáo trộn, nhằm bảo toàn tính tuần tự của chuỗi.

Xây dựng và huấn luyện mô hình SVR:

SVR với kernel RBF được sử dụng, tham số C và epsilon được điều chỉnh (trong ví dụ C=100, epsilon=0.01) để phù hợp với dữ liệu chuẩn hóa.

Mô hình được huấn luyện trên tập huấn luyện.

Dự báo, đánh giá và trực quan hóa:

Mô hình dự báo trên tập kiểm tra, MSE được tính để đánh giá hiệu năng.

Kết quả dự báo và giá trị thực được chuyển đổi về giá trị ban đầu (nghịch đảo quá trình chuẩn hóa) và vẽ biểu đồ so sánh.

